# V14 – Aufgaben: Musterlösungen

Vollständige Lösungen zu allen fünf Aufgaben aus [../03_aufgaben.ipynb](../03_aufgaben.ipynb). Bitte nur reinschauen, nachdem du die Aufgaben **selbst** versucht hast — sonst lernst du nichts.

Alle Lösungen nutzen die Testdaten aus dem Vorlesungsordner (daher das `../` im Pfad). Der Werkzeug-Ordner für Aufgabe 4 wird in diesem Ordner frisch angelegt und am Ende wieder gelöscht.


## Aufgabe 1 – CNC-Präzisions-Analyse

Klassisches CSV-Lese-Pattern: `with open(...)`, Kopfzeile mit `readline()` verwerfen, danach zeilenweise iterieren, Spalten mit `.split(",")` trennen und den Wert an Index 1 (Abweichung_X_um) aufsummieren. Zum Schluss Summe durch Anzahl teilen.


In [1]:
summe = 0.0
anzahl = 0
with open("../cnc_praezision.csv", encoding="utf-8") as f:
    f.readline()  # Kopfzeile ueberspringen
    for zeile in f:
        spalten = zeile.strip().split(",")
        if len(spalten) < 2:
            continue
        summe += float(spalten[1])
        anzahl += 1

durchschnitt = summe / anzahl
print(f"Durchschnittliche X-Abweichung: {durchschnitt:.2f} µm (aus {anzahl} Maschinen)")
assert round(durchschnitt, 2) == 5.02
print("✅ Aufgabe 1 korrekt.")


Durchschnittliche X-Abweichung: 5.02 µm (aus 15 Maschinen)
✅ Aufgabe 1 korrekt.


## Aufgabe 2 – Hydraulik-Durchfluss

Einfachste Lösung: eingebautes `max(werte)`. Alternativ: Schleife mit Vergleich — bildet den Algorithmus aber nur nach.


In [2]:
messwerte = [42.1, 38.5, 47.8, 39.2, 45.0, 41.3, 44.7]

def max_durchfluss(werte):
    return max(werte)

# Alternative ohne eingebautes max:
def max_durchfluss_manuell(werte):
    groesster = werte[0]
    for w in werte[1:]:
        if w > groesster:
            groesster = w
    return groesster

spitze = max_durchfluss(messwerte)
print(f"Spitzen-Durchfluss: {spitze} l/min")
assert spitze == 47.8
assert max_durchfluss_manuell([1.0, 5.5, 2.2]) == 5.5
print("✅ Aufgabe 2 korrekt.")


Spitzen-Durchfluss: 47.8 l/min
✅ Aufgabe 2 korrekt.


## Aufgabe 3 – Material-Festigkeits-Vergleich

`json.load(f)` liefert uns das Dictionary direkt, danach iterieren wir über `daten["proben"]` und merken uns den Eintrag mit dem größten `zugfestigkeit_mpa`.


In [3]:
import json

with open("../werkstoff_pruefung.json", encoding="utf-8") as f:
    daten = json.load(f)

bestes_material = ""
beste_zugfestigkeit = 0
for probe in daten["proben"]:
    if probe["zugfestigkeit_mpa"] > beste_zugfestigkeit:
        beste_zugfestigkeit = probe["zugfestigkeit_mpa"]
        bestes_material = probe["werkstoff"]

print(f"Werkstoff mit hoechster Zugfestigkeit: {bestes_material} ({beste_zugfestigkeit} MPa)")
assert bestes_material == "42CrMo4"
assert beste_zugfestigkeit == 1100
print("✅ Aufgabe 3 korrekt.")


Werkstoff mit hoechster Zugfestigkeit: 42CrMo4 (1100 MPa)
✅ Aufgabe 3 korrekt.


## Aufgabe 4 – Werkzeug-Ordner-Übersicht

Damit das Notebook alleine lauffähig ist, legen wir den Werkzeug-Ordner am Anfang selbst an. Die eigentliche Lösungs-Funktion filtert mit `os.path.isfile` aus `os.listdir` heraus, was wirklich eine Datei ist, und baut den Pfad mit `os.path.join` plattformunabhängig auf.


In [4]:
import os

# Setup im loesungen/-Ordner, damit kein Pfad nach aussen noetig ist.
werkzeug_ordner = "werkzeug_ordner"
os.makedirs(werkzeug_ordner, exist_ok=True)
for name, inhalt in {
    "fraeser_8mm.txt":   "Typ: Schaftfraeser\n",
    "bohrer_6mm.txt":    "HSS-Co, D=6mm\n",
    "gewindeschneider.txt": "M10x1.5\n",
    "drehmeissel.txt":   "CCMT 09T304\n",
}.items():
    with open(os.path.join(werkzeug_ordner, name), "w", encoding="utf-8", newline="\n") as f:
        f.write(inhalt)
os.makedirs(os.path.join(werkzeug_ordner, "archiv_alte_werkzeuge"), exist_ok=True)

def liste_dateien(ordner):
    ergebnis = []
    for name in os.listdir(ordner):
        if os.path.isfile(os.path.join(ordner, name)):
            ergebnis.append(name)
    return sorted(ergebnis)

werkzeuge = liste_dateien(werkzeug_ordner)
print("Dateien:", werkzeuge)
assert werkzeuge == ["bohrer_6mm.txt", "drehmeissel.txt", "fraeser_8mm.txt", "gewindeschneider.txt"]
print("✅ Aufgabe 4 korrekt.")


Dateien: ['bohrer_6mm.txt', 'drehmeissel.txt', 'fraeser_8mm.txt', 'gewindeschneider.txt']
✅ Aufgabe 4 korrekt.


## Aufgabe 5 – Produktionslinien-Vergleich

Wir parsen die XML mit `xml.etree.ElementTree`, iterieren über alle `linie`-Elemente, berechnen die Auslastung als `output / kapazitaet * 100` und merken uns das Maximum. Am Ende räumen wir den Werkzeug-Ordner wieder auf.


In [5]:
import os
import shutil
import xml.etree.ElementTree as ET

tree = ET.parse("../produktionslinien_vergleich.xml")
root = tree.getroot()

beste_linie = ""
beste_quote = 0.0

for linie in root.findall("linien/linie"):
    name = linie.find("name").text
    kapazitaet = float(linie.find("kapazitaet_stueck_h").text)
    output = float(linie.find("tatsaechliche_output_stueck_h").text)
    auslastung = output / kapazitaet * 100
    print(f"  {name:8s} {auslastung:5.1f} %")
    if auslastung > beste_quote:
        beste_quote = auslastung
        beste_linie = name

print(f"\nBeste Linie: {beste_linie} mit {beste_quote:.1f} %")
assert beste_linie == "Linie-D"
assert abs(beste_quote - 95.0) < 0.05
print("✅ Aufgabe 5 korrekt.")

# Aufraeumen
if os.path.exists("werkzeug_ordner"):
    shutil.rmtree("werkzeug_ordner")
    print("Werkzeug-Ordner entfernt.")


  Linie-A   90.0 %
  Linie-B   91.8 %
  Linie-C   94.7 %
  Linie-D   95.0 %
  Linie-E   92.5 %

Beste Linie: Linie-D mit 95.0 %
✅ Aufgabe 5 korrekt.
Werkzeug-Ordner entfernt.
